# Step 6 — Predictor 完整 held-out evaluation

在 validation/test split 上抽取生成进度 25%、50%、75%、100% 四个位置的联合分布，评估：

- Correctness：AUROC、AUPRC、accuracy、F1、incorrect recall、Brier、ECE；
- Remaining length：MAE、median AE、预测值 vs 真值；
- reliability diagram，以及越接近结尾是否越会判断正确性。

本步骤默认评估最终 ZIP-RC 模型；模型从未用 validation/test trajectory 更新参数。

In [ ]:
from pathlib import Path
import sys

candidates = [Path.cwd(), *Path.cwd().parents, Path("/content/ZIP-RC")]
REPO = next(
    (path for path in candidates if (path / "notebooks" / "ziprc_notebook_utils.py").exists()),
    None,
)
if REPO is None:
    raise FileNotFoundError("找不到 ZIP-RC 仓库；请从仓库根目录或 notebooks/ 运行。")

sys.path.insert(0, str(REPO / "notebooks"))
from ziprc_notebook_utils import *

CONFIG = load_config(REPO)
print("Repository:", REPO)
print("Experiment:", CONFIG["experiment_name"])

In [ ]:
import ast
import json

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from IPython.display import display
from sklearn.metrics import accuracy_score, average_precision_score, brier_score_loss, f1_score, recall_score, roc_auc_score
from transformers import AutoModelForCausalLM

final_model = REPO / CONFIG["paths"]["final_model"]
eval_sources = {
    "validation": REPO / CONFIG["paths"]["validation"],
    "test": REPO / CONFIG["paths"]["test"],
}
positions_path = REPO / CONFIG["paths"]["predictor_positions"]
metrics_path = REPO / CONFIG["paths"]["predictor_metrics"]

def as_int_list(value):
    if isinstance(value, str):
        return [int(item) for item in ast.literal_eval(value)]
    return [int(item) for item in value]

def expected_calibration_error(labels, probabilities, bins=10):
    labels = np.asarray(labels, dtype=float)
    probabilities = np.asarray(probabilities, dtype=float)
    edges = np.linspace(0, 1, bins + 1)
    result = 0.0
    for left, right in zip(edges[:-1], edges[1:]):
        mask = (probabilities >= left) & (probabilities < right if right < 1 else probabilities <= right)
        if mask.any():
            result += mask.mean() * abs(labels[mask].mean() - probabilities[mask].mean())
    return float(result)

In [ ]:
RUN_EVALUATION = True
if RUN_EVALUATION:
    device = torch.device("cuda:0")
    dtype = torch.bfloat16
    model = AutoModelForCausalLM.from_pretrained(
        final_model, torch_dtype=dtype, trust_remote_code=True, attn_implementation="flash_attention_2"
    ).to(device).eval()
    frames = []
    for split_name, split_path in eval_sources.items():
        split_frame = pd.read_parquet(split_path).copy()
        split_frame["eval_split"] = split_name
        frames.append(split_frame)
    frame = pd.concat(frames, ignore_index=True)
    reward_values = torch.tensor(CONFIG["reward_values"], dtype=torch.float32, device=device)
    length_edges = np.array([0, 256, 512, 1024, 2048, 4096, 8192, 16384, 32768])
    length_midpoints = torch.tensor((length_edges[:-1] + length_edges[1:]) / 2, dtype=torch.float32, device=device)
    lm_head = model.get_output_embeddings()
    num_reward_states = len(CONFIG["reward_values"])
    num_length_bins = int(CONFIG["num_length_bins"])
    start = int(CONFIG["distribution_token_id"])
    weight = lm_head.weight[start:start + num_reward_states * num_length_bins]
    bias = lm_head.bias[start:start + num_reward_states * num_length_bins] if getattr(lm_head, "bias", None) is not None else None

    rows = []
    with torch.inference_mode():
        for row_idx, row in frame.iterrows():
            ids = as_int_list(row["input_ids"])[:-1][:int(CONFIG["train_max_length"])]
            label_positions = [position - 1 for position in as_int_list(row["label_positions"]) if 0 <= position - 1 < len(ids)]
            if not label_positions:
                continue
            inputs = torch.tensor(ids, dtype=torch.long, device=device).unsqueeze(0)
            hidden = model(input_ids=inputs, output_hidden_states=True, use_cache=False).hidden_states[-1][0]
            for progress in (.25, .50, .75, 1.0):
                list_index = min(len(label_positions) - 1, max(0, round(progress * len(label_positions)) - 1))
                position = label_positions[list_index]
                logits = F.linear(hidden[position], weight, bias).float()
                probs = F.softmax(logits, dim=-1).view(num_reward_states, num_length_bins)
                reward_probs = probs.sum(dim=1)
                length_probs = probs.sum(dim=0)
                rows.append({
                    "row_idx": row_idx,
                    "prompt_idx": int(row["prompt_idx"]),
                    "eval_split": row["eval_split"],
                    "progress": progress,
                    "correct": bool(row["correct"]),
                    "predicted_reward": float(torch.dot(reward_probs, reward_values).item()),
                    "predicted_remaining": float(torch.dot(length_probs, length_midpoints).item()),
                    "true_remaining": int(label_positions[-1] - position),
                    "position": int(position),
                })
    del model
    torch.cuda.empty_cache()
    position_df = pd.DataFrame(rows)
    positions_path.parent.mkdir(parents=True, exist_ok=True)
    position_df.to_parquet(positions_path, index=False)
else:
    position_df = pd.read_parquet(positions_path)
print("Rows:", len(position_df), "saved to", positions_path)

In [ ]:
metric_rows = []
for (split_name, progress), group in position_df.groupby(["eval_split", "progress"]):
    labels = group["correct"].astype(int)
    scores = group["predicted_reward"].clip(0, 1)
    predictions = scores >= .5
    errors = (group["predicted_remaining"] - group["true_remaining"]).abs()
    naive_remaining = float(group["true_remaining"].median())
    naive_errors = (group["true_remaining"] - naive_remaining).abs()
    metric_rows.append({
        "eval_split": split_name,
        "progress": progress,
        "auroc": roc_auc_score(labels, scores) if labels.nunique() == 2 else np.nan,
        "auprc": average_precision_score(labels, scores) if labels.nunique() == 2 else np.nan,
        "accuracy": accuracy_score(labels, predictions),
        "f1": f1_score(labels, predictions, zero_division=0),
        "incorrect_recall": recall_score(labels, predictions, pos_label=0, zero_division=0),
        "brier": brier_score_loss(labels, scores),
        "ece": expected_calibration_error(labels, scores),
        "length_mae": errors.mean(),
        "length_median_ae": errors.median(),
        "length_naive_mae": naive_errors.mean(),
    })
metrics = pd.DataFrame(metric_rows).sort_values(["eval_split", "progress"])
metrics_path.write_text(metrics.to_json(orient="records", indent=2), encoding="utf-8")
display(metrics.round(4))

fig, axes = plt.subplots(2, 2, figsize=(13, 9))
test_metrics = metrics[metrics["eval_split"] == "test"]
test_metrics.plot(x="progress", y=["auroc", "auprc"], marker="o", ax=axes[0, 0], ylim=(0, 1))
axes[0, 0].axhline(.5, color="gray", linestyle="--")
axes[0, 0].set_title("Correctness discrimination by progress")
test_metrics.plot(x="progress", y=["incorrect_recall", "f1"], marker="o", ax=axes[0, 1], ylim=(0, 1))
axes[0, 1].set_title("Error detection / F1")
test_metrics.plot(x="progress", y=["length_mae", "length_median_ae", "length_naive_mae"], marker="o", ax=axes[1, 0])
axes[1, 0].set_title("Remaining-length error")

final = position_df[(position_df["eval_split"] == "test") & (position_df["progress"] == 1.0)].copy()
final["calibration_bin"] = pd.cut(final["predicted_reward"], bins=np.linspace(0, 1, 11), include_lowest=True)
calibration = final.groupby("calibration_bin", observed=False).agg(predicted=("predicted_reward", "mean"), observed=("correct", "mean"), count=("correct", "size")).dropna()
axes[1, 1].plot([0, 1], [0, 1], color="gray", linestyle="--")
axes[1, 1].plot(calibration["predicted"], calibration["observed"], marker="o")
axes[1, 1].set(xlim=(0, 1), ylim=(0, 1), title="Reliability diagram @100%", xlabel="predicted", ylabel="observed")
plt.tight_layout()
plt.show()

final_metrics = test_metrics.iloc[-1]
early_length = test_metrics[test_metrics["progress"] < 1.0]
length_has_signal = bool((early_length["length_mae"] < early_length["length_naive_mae"]).any())
checks = [
    gate("Validation / test 均齐全", set(position_df["eval_split"]) == {"validation", "test"}, str(position_df['eval_split'].value_counts().to_dict())),
    gate("四个进度点齐全", set(position_df["progress"]) == {.25, .5, .75, 1.0}, str(sorted(position_df['progress'].unique()))),
    gate("预测值有效", position_df[["predicted_reward", "predicted_remaining"]].notna().all().all(), "no missing values"),
    gate("AUROC >0.55", final_metrics["auroc"] > .55, f"{final_metrics['auroc']:.3f}", kind="scientific"),
    gate("Incorrect recall ≥0.50", final_metrics["incorrect_recall"] >= .50, f"{final_metrics['incorrect_recall']:.3f}", kind="scientific"),
    gate("Remaining length 优于 naive", length_has_signal, "25%/50%/75% 至少一个进度点 MAE 更低", kind="scientific"),
]
display(gate_frame(checks))
save_stage_report(REPO, "06_predictor_evaluation", checks, {"metrics": metrics.to_dict(orient="records")})